# Extension Step 4 — GNN (DAGNN) Task Graph Classifier

Trains a `DAGClassifier` on task graph realizations using the official
CaptainCook4D train/val/test split from `extension/step1/combined_recordings.json`.

The val set drives LR scheduling and checkpoint selection.
The test set is evaluated once at the end on the best-val-AUC checkpoint.

**Prerequisites:**
- Step embeddings in `STEP_EMBEDDINGS_DIR` (output of Extension Step 1)
- Pre-fusion cache in `CACHE_DIR` (output of b3_hungarian_matching.ipynb, sim_threshold=0.30)
- Task graphs in `GRAPHS_DIR` (annotations submodule)
- EgoVLP checkpoint at `EGOVLP_CKPT`
- Official split at `SPLITS_JSON` (`extension/step1/combined_recordings.json`)

**Output:**
- Best checkpoint in `STEP4_OUTPUT_DIR/checkpoints/best.pt`
- Split assignment in `STEP4_OUTPUT_DIR/split_info.json`
- Metrics in `STEP4_OUTPUT_DIR/results.csv` (val row + test row)

In [1]:
# ── 1. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [2]:
# -- 2. Path constants + split mode -------------------------------------------------

# CHOOSE YOUR SPLIT MODE
#   "official" -- official CaptainCook4D recording-level split (combined_recordings.json)
#   "recipe"   -- 16 train / 4 val / 4 test recipes, seed=42
#   "loo"      -- Leave-One-Out cross-validation (one fold per recipe, same protocol as B2)
SPLIT_MODE = "loo"

DRIVE_ROOT           = '/content/drive/MyDrive/AML_Project'
REPO_DIR             = '/content/code'
EGOVLP_REPO          = '/content/EgoVLP'

EGOVLP_CKPT          = f'{DRIVE_ROOT}/models/egovlp.pth'
STEP_EMBEDDINGS_DIR  = f'{DRIVE_ROOT}/step1/step_embeddings'
ANNOTATIONS_PATH     = f'{REPO_DIR}/annotations/annotation_json/complete_step_annotations.json'
GRAPHS_DIR           = f'{REPO_DIR}/annotations/task_graphs'
SPLITS_JSON          = f'{REPO_DIR}/extension/step1/combined_recordings.json'

# Pre-fusion cache built by b3_hungarian_matching.ipynb (sim_threshold=0.30)
CACHE_DIR            = f'{DRIVE_ROOT}/step3/cache_thr030'
STEP4_OUTPUT_DIR     = f'{DRIVE_ROOT}/step4/results_{SPLIT_MODE}'

# local copy of step embeddings (faster I/O than Drive during training)
LOCAL_EMBEDDINGS_DIR = '/content/step_embeddings'

print(f'split mode : {SPLIT_MODE}')
print(f'cache dir  : {CACHE_DIR}')
print(f'output dir : {STEP4_OUTPUT_DIR}')

split mode : loo
cache dir  : /content/drive/MyDrive/AML_Project/step3/cache_thr030
output dir : /content/drive/MyDrive/AML_Project/step4/results_loo


In [3]:
# ── 3. Clone repos (--recursive fetches annotations submodule) ────────────────
!git clone --recursive https://github.com/Laio95/aml-2025-mistake-detection.git {REPO_DIR}
!git clone https://github.com/showlab/EgoVLP.git {EGOVLP_REPO}

Cloning into '/content/code'...
remote: Enumerating objects: 724, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 724 (delta 136), reused 131 (delta 128), pack-reused 543 (from 2)
Receiving objects: 100% (724/724), 3.70 MiB | 8.59 MiB/s, done.
Resolving deltas: 100% (449/449), done.
Submodule 'actionformer_release' (https://github.com/rohithpeddi/actionformer_release.git) registered for path 'actionformer_release'
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/actionformer_release'...
remote: Enumerating objects: 410, done.        
remote: Counting objects: 100% (26/26), done.        
remote: Compressing objects: 100% (20/20), done.        
remote: Total 410 (delta 16), reused 6 (delta 6), pack-reused 384 (from 2)        
Receiving objects: 100% (410/410), 651.07 KiB | 3.58 MiB/s, done.
Resolving deltas: 100% (225/225), done.
Cloning

In [4]:
import torch, os
pt_version  = torch.__version__.split('+')[0]
os.environ['TORCH'] = pt_version

cuda_version = torch.version.cuda
if cuda_version is not None:
    cuda_str = f"cu{cuda_version.replace('.', '')}"
    os.environ['CUDA'] = cuda_str
    cuda_suffix = f"+{cuda_str}"
    print(f'PyTorch {pt_version}, CUDA {cuda_str}')
else:
    # CUDA is not available or not detected by PyTorch
    os.environ['CUDA'] = "" # Set CUDA env var to empty string
    cuda_suffix = "" # No CUDA suffix for URL
    print(f'PyTorch {pt_version}, CUDA not available. Dependencies will be installed for CPU.')

# torch-scatter / torch-sparse need precompiled binaries matching the runtime
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}{cuda_suffix}.html -q
!pip install decord pytorchvideo fvcore iopath torch-geometric -q
!pip install -r {REPO_DIR}/requirements.txt -q
print('Dependencies installed.')

PyTorch 2.10.0, CUDA cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 423, in run
    _, build_failures = build(
                        ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/wheel_builder.py", line 319, in build
    wheel_file = _build_one(
               

In [9]:
# TODO: se quella sopra va bene cancellare questa cella

# Prova di risoluzione alla cella sopra
import torch
import os

# 1. Recupera la versione di PyTorch e CUDA attualmente attive
pt_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda
cuda_version_str = f"cu{cuda_version.replace('.', '')}"

# Salva le variabili nell'ambiente per poterle usare nei comandi bash (!)
os.environ['TORCH'] = pt_version
os.environ['CUDA'] = cuda_version_str

print(f"Sto per installare i binari per PyTorch {pt_version} e CUDA {cuda_version_str}")

# 2. Installa torch-scatter e torch-sparse FORZANDO l'uso dei binari precompilati
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}+${CUDA}.html

# 3. Installa il resto delle librerie in modo standard
!pip install decord ffmpeg pytorchvideo fvcore iopath torch-geometric
!pip install -r /content/code/requirements.txt

Sto per installare i binari per PyTorch 2.10.0 e CUDA cu128
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 115.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 137.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for ffmpeg: filename=ffmpeg-1.4-py3-none-any.whl size=6083 sha256=c0cb9e1ffb3bf7a4d839c0b861d8a96f637533b11d84b185021dcb4c4bf555b6
  Stored in directory: /root/.cache/pip/wheels/26/21/0c/c26e09dff860a9071683e279445262346e008a9a1d2142c4ad
Successfully built ffmpeg
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118


In [5]:
# ── 5. Copy step embeddings to local storage (faster than reading from Drive) ──
import os
os.makedirs(LOCAL_EMBEDDINGS_DIR, exist_ok=True)
!cp -r {STEP_EMBEDDINGS_DIR}/* {LOCAL_EMBEDDINGS_DIR}/
print(f"Copied: {len(os.listdir(LOCAL_EMBEDDINGS_DIR))} files → {LOCAL_EMBEDDINGS_DIR}")

Copied: 384 files → /content/step_embeddings


In [6]:
# ── 6. Verify pre-fusion cache (built by b3_hungarian_matching.ipynb) ──────
# The cache was built with sim_threshold=SIM_THRESHOLD by b3_hungarian_matching.
# Do NOT clear it — clearing would force a rebuild with random projector weights.
# To rebuild with a different threshold, re-run hungarian_matching.ipynb.
import os, pathlib
cache_pt = list(pathlib.Path(CACHE_DIR).glob("*.pt")) if os.path.exists(CACHE_DIR) else []
if cache_pt:
    print(f"Cache ready: {len(cache_pt)} files in {CACHE_DIR}")
else:
    print(f"WARNING: cache not found at {CACHE_DIR}")
    print("Run hungarian_matching.ipynb first.")

Cache ready: 384 files in /content/drive/MyDrive/AML_Project/step3/cache_thr030


In [7]:
# EgoVLP's FrozenInTime constructor loads a local ViT-B/16 checkpoint from
# /content/egovlp/pretrained/jx_vit_base_p16_224-80ecf9dd.pth. Download it.
!mkdir -p {EGOVLP_REPO}/pretrained
!wget -q -nc -O {EGOVLP_REPO}/pretrained/jx_vit_base_p16_224-80ecf9dd.pth \
  https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
!ls -la {EGOVLP_REPO}/pretrained/

total 338188
drwxr-xr-x  2 root root      4096 May 17 16:52 .
drwxr-xr-x 13 root root      4096 May 17 16:52 ..
-rw-r--r--  1 root root 346292833 Dec  7  2021 jx_vit_base_p16_224-80ecf9dd.pth


In [8]:
# ── 7. WandB login ────────────────────────────────────────────────────────────
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: albertogiunti2001 (albertogiunti2001-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Smoke test — 3 epochs (sanity check)

Verify that:
- The pre-fusion cache is complete (384 `.pt` files)
- The `DAGClassifier` (DAGNN) runs without errors on the train/val/test split
- Loss and val AUC have sensible values after 3 epochs

**Estimated time:** ~1-2 min.

In [10]:
SMOKE_OUTPUT_DIR = f'{DRIVE_ROOT}/step4/results_smoke_thr030'

In [ ]:
'''
%%bash -s "$REPO_DIR" "$ANNOTATIONS_PATH" "$LOCAL_EMBEDDINGS_DIR" "$GRAPHS_DIR" "$EGOVLP_REPO" "$EGOVLP_CKPT" "$CACHE_DIR" "$SMOKE_OUTPUT_DIR" "$SPLITS_JSON"

cd /content/code

python -m extension.step4.train_dag_classifier \
    --annotations_path    "$2" \
    --step_embeddings_dir "$3" \
    --graphs_dir          "$4" \
    --egovlp_repo         "$5" \
    --egovlp_ckpt         "$6" \
    --cache_dir           "$7" \
    --output_dir          "$8" \
    --splits_json         "$9" \
    --num_epochs   3  \
    --hidden_dim   128 \
    --num_layers   2  \
    --dropout      0.5 \
    --lr           1e-3 \
    --weight_decay 1e-4 \
    --batch_size   4  \
    --threshold    0.5 \
    --seed         42  \
    --num_workers  2 \
    --enable_wandb
'''

## Full training — 50 epochs

Trains the `DAGClassifier` (DAGNN) with the split selected in `SPLIT_MODE` (cell 2).

### Split modes

| `SPLIT_MODE` | Train | Val | Test | Protocol |
|---|---|---|---|---|
| `"official"` | 213 recordings | 62 recordings | 109 recordings | Official CaptainCook4D split (`combined_recordings.json`) — recording-level |
| `"recipe"` | 16 recipes (~281 rec.) | 4 recipes (~53 rec.) | 4 recipes (~50 rec.) | Recipe-level split, seed=42 — avoids data leakage across recipes |
| `"loo"` | K-1 recipes (~368 rec.) | — | 1 recipe (~16 rec.) | Leave-One-Out, 24 folds — same protocol as B2 (TaskVerifier) |

**Note:**
- `"official"`: val AUC drives LR scheduler + checkpoint selection; test evaluated once at the end.
- `"recipe"`: same logic as `"official"`, split at recipe level instead of recording level.
- `"loo"`: no val set; scheduler driven by AUC on the test fold; best checkpoint per fold saved to `checkpoints/fold_{i}_recipe_{id}_best.pt`; aggregated results in `results.csv`.

**Estimated time:** ~15-30 min on T4 for `"official"`/`"recipe"`; ~6-10 hours for `"loo"` (24 folds × 50 epochs).

In [11]:
import subprocess, sys, os

os.chdir('/content/code')

common = [
    "--annotations_path",    ANNOTATIONS_PATH,
    "--step_embeddings_dir", LOCAL_EMBEDDINGS_DIR,
    "--graphs_dir",          GRAPHS_DIR,
    "--egovlp_repo",         EGOVLP_REPO,
    "--egovlp_ckpt",         EGOVLP_CKPT,
    "--cache_dir",           CACHE_DIR,
    "--output_dir",          STEP4_OUTPUT_DIR,
    "--num_epochs",  "50",
    "--hidden_dim",  "128",
    "--num_layers",  "2",
    "--dropout",     "0.5",
    "--lr",          "1e-3",
    "--weight_decay","1e-4",
    "--batch_size",  "4",
    "--threshold",   "0.5",
    "--seed",        "42",
    "--num_workers", "2",
    "--enable_wandb",
]

if SPLIT_MODE == "loo":
    cmd = [sys.executable, "-m", "extension.step4.train_dag_classifier",
           *common, "--split_mode", "loo"]
elif SPLIT_MODE == "recipe":
    cmd = [sys.executable, "-m", "extension.step4.train_dag_classifier",
           *common, "--split_mode", "recipe", "--recipe_split_seed", "42"]
else:  # official
    cmd = [sys.executable, "-m", "extension.step4.train_dag_classifier",
           *common, "--split_mode", "official", "--splits_json", SPLITS_JSON]

subprocess.run(cmd, check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'extension.step4.train_dag_classifier', '--annotations_path', '/content/code/annotations/annotation_json/complete_step_annotations.json', '--step_embeddings_dir', '/content/step_embeddings', '--graphs_dir', '/content/code/annotations/task_graphs', '--egovlp_repo', '/content/EgoVLP', '--egovlp_ckpt', '/content/drive/MyDrive/AML_Project/models/egovlp.pth', '--cache_dir', '/content/drive/MyDrive/AML_Project/step3/cache_thr030', '--output_dir', '/content/drive/MyDrive/AML_Project/step4/results_loo', '--num_epochs', '50', '--hidden_dim', '128', '--num_layers', '2', '--dropout', '0.5', '--lr', '1e-3', '--weight_decay', '1e-4', '--batch_size', '4', '--threshold', '0.5', '--seed', '42', '--num_workers', '2', '--enable_wandb', '--split_mode', 'loo'], returncode=0)

## Results — verification and summary


In [12]:
# ── Verify checkpoint and split ─────────────────────────────────────────────
import pathlib, torch, json

ckpt_path = pathlib.Path(STEP4_OUTPUT_DIR) / 'checkpoints' / 'best.pt'
if ckpt_path.exists():
    saved = torch.load(ckpt_path, weights_only=False)
    print(f"Checkpoint: {ckpt_path.name}")
    print(f"  Best epoch  : {saved['epoch']}")
    print(f"  Val AUC     : {saved['val_metrics']['auc']:.4f}")
    print(f"  Val F1      : {saved['val_metrics']['f1']:.4f}")
    print(f"  Val Accuracy: {saved['val_metrics']['accuracy']:.4f}")
else:
    print(f"WARNING: checkpoint not found at {ckpt_path}")

split_path = pathlib.Path(STEP4_OUTPUT_DIR) / 'split_info.json'
if split_path.exists():
    info = json.load(open(split_path))
    print(f"\nSplit (from {info['splits_json']})")
    print(f"  train : {len(info['train'])} recordings")
    print(f"  val   : {len(info['val'])} recordings")
    print(f"  test  : {len(info['test'])} recordings")

In [13]:
# ── Read and print results.csv ──────────────────────────────────────────────
import pandas as pd

csv_path = f'{STEP4_OUTPUT_DIR}/results.csv'
df = pd.read_csv(csv_path)
print(df.to_string(index=False))

fold  activity_id      auc       f1  accuracy
   1          1.0 0.723077 0.769231  0.666667
   2          2.0 0.916667 0.461538  0.562500
   3          3.0 0.600000 0.500000  0.538462
   4          4.0 0.771429 0.636364  0.529412
   5          5.0 0.571429 0.000000  0.533333
   6          7.0 0.650000 0.000000  0.375000
   7          8.0 0.833333 0.833333  0.750000
   8          9.0 0.777778 0.000000  0.642857
   9         10.0 0.781250 0.800000  0.750000
  10         12.0 0.766234 0.560000  0.388889
  11         13.0 0.822222 0.800000  0.785714
  12         15.0 0.720000 0.181818  0.400000
  13         16.0 0.700000 0.700000  0.625000
  14         17.0 0.843750 0.592593  0.450000
  15         18.0 0.795455 0.000000  0.266667
  16         20.0 0.979167 0.769231  0.785714
  17         21.0 0.726190 0.761905  0.736842
  18         22.0 0.772727 0.761905  0.705882
  19         23.0 0.800000 0.000000  0.625000
  20         25.0 0.767857 0.705882  0.666667
  21         26.0 0.885714 0.85714